# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` and other libraries are installed
!pip install mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset name and description
print(f"{metadata.name}\n{metadata.description}")

## 2. Data Overview
Review available record sets (tables), fields, their `@id`s, and columns.

Below we enumerate all Record Sets and their fields by their `@id` using `dataset.metadata`.

In [ ]:
# List all record sets and fields referenced by `@id`
import json

def print_record_sets_and_fields(md):
    """
    Print all RecordSets, Field `@id`s, and their associated Column `@id`s from the metadata.
    """
    if not hasattr(md, 'record_sets'):
        print("No record sets found in this dataset.")
        return

    for recset in md.record_sets:
        print(f'RecordSet: @id={recset.id}, name={getattr(recset, "name", "")}')
        if hasattr(recset, 'fields'):
            for field in recset.fields:
                print(f'  Field: @id={field.id}, name={getattr(field, "name", "")}, dataType={getattr(field, "data_type", "")}')
                colids = [c.id for c in getattr(field, 'columns', [])] if hasattr(field, 'columns') else []
                if colids:
                    print(f'    Columns: {colids}')
        print("")

print_record_sets_and_fields(metadata)

In some environments, or for inline review, you can also programmatically fetch a summary of all available RecordSets and their field names as lists for later extraction.

In [ ]:
# Gather all record set @id values for use in extraction
record_sets_ids = []
record_set_names = []
fields_for_recordset = {}

if hasattr(metadata, 'record_sets'):
    for record_set in metadata.record_sets:
        record_sets_ids.append(record_set.id)
        record_set_names.append(getattr(record_set, 'name', record_set.id))
        fields_for_recordset[record_set.id] = [(field.id, getattr(field, 'name', '')) for field in getattr(record_set, 'fields', [])]
print('RecordSet @id list:', record_sets_ids)
print('RecordSet names:', record_set_names)
# List one record set's fields as an example
if record_sets_ids:
    print(f"Fields for RecordSet {record_sets_ids[0]}:", fields_for_recordset[record_sets_ids[0]])

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use RecordSet and Field `@id`s from the overview above.

If the dataset has multiple record sets, we'll load one for demonstration, but you can extend this to others as needed.

In [ ]:
# Extract data from each record set by @id
dataframes = {}

for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set {record_set_id} with {len(df)} records and columns: {df.columns.tolist()[:8]}{'...' if len(df.columns)>8 else ''}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}:", e)

# Show preview of the first available record set
if record_sets_ids:
    preview_id = record_sets_ids[0]
    print(f"Columns: {dataframes[preview_id].columns.tolist()}")
    dataframes[preview_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note:** Make sure to substitute the `@id` of numeric and grouping fields using the output from the overview above.

In [ ]:
# Select a record set and field IDs for EDA
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Substitute with actual @id from your record set/overview!
record_set_id = record_sets_ids[0] if record_sets_ids else None

# Find a likely numeric field for demonstration
sample_df = dataframes[record_set_id] if record_set_id else pd.DataFrame()
numeric_field_id = None
group_field_id = None
for col in sample_df.columns:
    # Heuristic: choose first number-like 
    if pd.api.types.is_numeric_dtype(sample_df[col]):
        numeric_field_id = col
        break
# Choose a group field (for demonstration, often a text/label field)
for col in sample_df.columns:
    if col != numeric_field_id and sample_df[col].dtype == object:
        group_field_id = col
        break

print(f"Numeric field selected: {numeric_field_id}")
print(f"Grouping field selected: {group_field_id}")

if numeric_field_id is not None:
    # Remove NAs
    clean_df = sample_df.dropna(subset=[numeric_field_id])
    threshold = np.nanmean(clean_df[numeric_field_id]) if len(clean_df) > 0 else 0
    # Filter: values greater than mean
    filtered_df = clean_df[clean_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}, count: {len(filtered_df)}")
    # Normalization
    filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() + 1e-8)
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Grouping/aggregation
    if group_field_id is not None:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        grouped = grouped.rename(columns={numeric_field_id: f"{numeric_field_id}_mean"})
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of normalized numeric field
if numeric_field_id is not None and filtered_df.shape[0]>0:
    plt.figure(figsize=(6,4))
    sns.histplot(filtered_df[numeric_field_id + '_normalized'], bins=30, kde=True)
    plt.title(f'Histogram of normalized {numeric_field_id}')
    plt.xlabel(f'{numeric_field_id}_normalized')
    plt.ylabel('Count')
    plt.show()
    
    # If a grouping field exists, boxplot
    if group_field_id is not None:
        plt.figure(figsize=(8,4))
        top_groups = filtered_df[group_field_id].value_counts().index[:6]
        top_df = filtered_df[filtered_df[group_field_id].isin(top_groups)]
        sns.boxplot(x=group_field_id, y=numeric_field_id + '_normalized', data=top_df)
        plt.title(f'Boxplot of {numeric_field_id} by {group_field_id} (Top 6)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded the Croissant dataset and schema from the provided URL using `mlcroissant`, inspected its metadata, enumerated all record sets and fields by their `@id`, and loaded the records into pandas DataFrames for processing and exploratory analysis.

We demonstrated basic filtering, normalization, grouping, and visualizations based on selected numeric and categorical fields. This workflow can be adapted to focus on different entities within the dataset by adjusting the `@id` references to your analysis needs.

For further work, consider exporting the DataFrames, exploring more domain-specific aggregations, or integrating with downstream ML pipelines.